# Info

- Split and Move Dataset to `data/completed`
- This notebook reads the three CSV files in `data/sample`, validates the source data, copies images and masks, and generates the following package:

```bash
data/completed/
├── train/
├── valid/
├── test/
├── gt/
└── sample/
    ├── train.csv
    ├── valid.csv
    └── test.csv
```


# Lib


In [1]:
from pathlib import Path
import filecmp
import shutil
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
import warnings

warnings.filterwarnings("ignore")

In [2]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

# Config


In [3]:
SPLIT_NAMES = ("train", "valid", "test")
EXPECTED_ROWS = {"train": 700, "valid": 100, "test": 200}
EXPECTED_LABELED = {"labeled_1_4": 175, "labeled_1_2": 350}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

COMMON_COLUMNS = [
    "image_id",
    "image_path",
    "mask_path",
    "stratum",
    "has_non_neoplastic",
    "has_neoplastic",
    "width",
    "height",
    "background_pixels",
    "non_neo_pixels",
    "neo_pixels",
]
TRAIN_COLUMNS = COMMON_COLUMNS + ["labeled_1_4", "labeled_1_2"]
# False: if a file exists in the destination, it will not be replaced even if the source file is different.
ALLOW_REPLACE_DIFFERENT_FILES = False

# Path


In [4]:
PROJECT_ROOT = Path("../..")
DATA_DIR = PROJECT_ROOT / "data"
SOURCE_SAMPLE_DIR = DATA_DIR / "sample"
COMPLETED_DIR = DATA_DIR / "completed"
COMPLETED_SAMPLE_DIR = COMPLETED_DIR / "sample"
COMPLETED_GT_DIR = COMPLETED_DIR / "gt"
COMPLETED_IMAGE_DIRS = {name: COMPLETED_DIR / name for name in SPLIT_NAMES}

print(f"PROJECT_ROOT          : {PROJECT_ROOT.resolve()}")
print(f"DATA_DIR              : {DATA_DIR.resolve()}")
print(f"SOURCE_SAMPLE_DIR     : {SOURCE_SAMPLE_DIR.resolve()}")
print(f"COMPLETED_DIR         : {COMPLETED_DIR.resolve()}")
print(f"COMPLETED_SAMPLE_DIR  : {COMPLETED_SAMPLE_DIR.resolve()}")
print(f"COMPLETED_GT_DIR      : {COMPLETED_GT_DIR.resolve()}")

PROJECT_ROOT          : B:\_projects\poply-image-segmentation
DATA_DIR              : B:\_projects\poply-image-segmentation\data
SOURCE_SAMPLE_DIR     : B:\_projects\poply-image-segmentation\data\sample
COMPLETED_DIR         : B:\_projects\poply-image-segmentation\data\completed
COMPLETED_SAMPLE_DIR  : B:\_projects\poply-image-segmentation\data\completed\sample
COMPLETED_GT_DIR      : B:\_projects\poply-image-segmentation\data\completed\gt


# Load sample csv


In [5]:
source_csv_paths = {name: SOURCE_SAMPLE_DIR / f"{name}.csv" for name in SPLIT_NAMES}
splits = {name: pd.read_csv(path) for name, path in source_csv_paths.items()}

for name, df in splits.items():
    expected_columns = TRAIN_COLUMNS if name == "train" else COMMON_COLUMNS
    if list(df.columns) != expected_columns:
        raise ValueError(
            f"{name}.csv wrong schema.\nExpected: {expected_columns}\nActual: {list(df.columns)}"
        )
    if len(df) != EXPECTED_ROWS[name]:
        raise ValueError(
            f"{name}.csv: wrong number of rows. Expected: {EXPECTED_ROWS[name]}, Actual: {len(df)}."
        )
    if df.isna().any().any():
        raise ValueError(f"{name}.csv has missing values.")

schema_summary = pd.DataFrame(
    [
        {
            "split": name,
            "rows": len(df),
            "columns": df.shape[1],
            "schema_ok": list(df.columns)
            == (TRAIN_COLUMNS if name == "train" else COMMON_COLUMNS),
            "missing_values": int(df.isna().sum().sum()),
        }
        for name, df in splits.items()
    ]
).set_index("split")

display(schema_summary)
for name, df in splits.items():
    display(Markdown(f"### `{name}.csv`"))
    display(df.head())

,rows,columns,schema_ok,missing_values
split,,,,
train,700,13,True,0
valid,100,11,True,0
test,200,11,True,0


### `train.csv`

,image_id,image_path,mask_path,stratum,has_non_neoplastic,has_neoplastic,width,height,background_pixels,non_neo_pixels,neo_pixels,labeled_1_4,labeled_1_2
0,0081835cf877e004e8bfb905b78a9139.jpeg,train/train/0081835cf877e004e8bfb905b78a9139.jpeg,train_gt/train_gt/0081835cf877e004e8bfb905b78a9139.jpeg,neo_only,0,1,1280,995,1252742,0,20858,0,1
1,00f9b6c911c4aa179344271fa54aa1b2.jpeg,train/train/00f9b6c911c4aa179344271fa54aa1b2.jpeg,train_gt/train_gt/00f9b6c911c4aa179344271fa54aa1b2.jpeg,neo_only,0,1,1280,959,1173318,0,54202,1,1
2,00fd197cd955fa095f978455cef3593c.jpeg,train/train/00fd197cd955fa095f978455cef3593c.jpeg,train_gt/train_gt/00fd197cd955fa095f978455cef3593c.jpeg,non_neo_only,1,0,1280,977,1143416,107144,0,0,0
3,01c13a3d810b000db55759486b8aa1b8.jpeg,train/train/01c13a3d810b000db55759486b8aa1b8.jpeg,train_gt/train_gt/01c13a3d810b000db55759486b8aa1b8.jpeg,non_neo_only,1,0,1280,995,1244543,29057,0,0,1
4,01e80947f9c02314fc58bdb26679bff5.jpeg,train/train/01e80947f9c02314fc58bdb26679bff5.jpeg,train_gt/train_gt/01e80947f9c02314fc58bdb26679bff5.jpeg,neo_only,0,1,1280,995,1161474,0,112126,0,1


### `valid.csv`

,image_id,image_path,mask_path,stratum,has_non_neoplastic,has_neoplastic,width,height,background_pixels,non_neo_pixels,neo_pixels
0,0496b83ef461c2a337948a41964c1d4f.jpeg,train/train/0496b83ef461c2a337948a41964c1d4f.jpeg,train_gt/train_gt/0496b83ef461c2a337948a41964c1d4f.jpeg,non_neo_only,1,0,1280,995,1237881,35719,0
1,051fc90e85d064f787d4be168d951058.jpeg,train/train/051fc90e85d064f787d4be168d951058.jpeg,train_gt/train_gt/051fc90e85d064f787d4be168d951058.jpeg,neo_only,0,1,1280,977,1225389,0,25171
2,07907941a8490fd7df8e26031fbb5e52.jpeg,train/train/07907941a8490fd7df8e26031fbb5e52.jpeg,train_gt/train_gt/07907941a8490fd7df8e26031fbb5e52.jpeg,neo_only,0,1,1280,995,1256778,0,16822
3,0bbd4ce2a1405f7d507592b964b7955a.jpeg,train/train/0bbd4ce2a1405f7d507592b964b7955a.jpeg,train_gt/train_gt/0bbd4ce2a1405f7d507592b964b7955a.jpeg,neo_only,0,1,1280,995,1233760,0,39840
4,0c90be319e1d6d19a707a4fdcc58c9c9.jpeg,train/train/0c90be319e1d6d19a707a4fdcc58c9c9.jpeg,train_gt/train_gt/0c90be319e1d6d19a707a4fdcc58c9c9.jpeg,non_neo_only,1,0,1280,977,1225582,24978,0


### `test.csv`

,image_id,image_path,mask_path,stratum,has_non_neoplastic,has_neoplastic,width,height,background_pixels,non_neo_pixels,neo_pixels
0,00d827b8bcf15050fb9af0ef92846b60.jpeg,train/train/00d827b8bcf15050fb9af0ef92846b60.jpeg,train_gt/train_gt/00d827b8bcf15050fb9af0ef92846b60.jpeg,neo_only,0,1,1280,995,1197272,0,76328
1,01ff069910ea2abe3c84417fda801941.jpeg,train/train/01ff069910ea2abe3c84417fda801941.jpeg,train_gt/train_gt/01ff069910ea2abe3c84417fda801941.jpeg,neo_only,0,1,1280,995,1261623,0,11977
2,040781d54ffd6a8ee924dbd70f618d52.jpeg,train/train/040781d54ffd6a8ee924dbd70f618d52.jpeg,train_gt/train_gt/040781d54ffd6a8ee924dbd70f618d52.jpeg,neo_only,0,1,1280,995,1212723,0,60877
3,081835cf877e004e8bfb905b78a91391.jpeg,train/train/081835cf877e004e8bfb905b78a91391.jpeg,train_gt/train_gt/081835cf877e004e8bfb905b78a91391.jpeg,neo_only,0,1,1280,995,1237606,0,35994
4,099130d98309817287a8aef566999467.jpeg,train/train/099130d98309817287a8aef566999467.jpeg,train_gt/train_gt/099130d98309817287a8aef566999467.jpeg,neo_only,0,1,1280,995,1197251,0,76349


# Check mapping, path, leakage


In [6]:
source_records = []
all_ids = {}

for split_name, df in splits.items():
    all_ids[split_name] = set(df["image_id"].astype(str))

    for row in df.itertuples(index=False):
        image_id = str(row.image_id)
        image_rel = Path(str(row.image_path))
        mask_rel = Path(str(row.mask_path))
        image_src = DATA_DIR / image_rel
        mask_src = DATA_DIR / mask_rel

        mapping_ok = image_rel.name == mask_rel.name == image_id
        source_records.append(
            {
                "split": split_name,
                "image_id": image_id,
                "mapping_ok": mapping_ok,
                "image_exists": image_src.is_file(),
                "mask_exists": mask_src.is_file(),
                "image_source": image_src,
                "mask_source": mask_src,
            }
        )

source_check = pd.DataFrame(source_records)

if not source_check["mapping_ok"].all():
    display(source_check.loc[~source_check["mapping_ok"]].head(20))
    raise ValueError("image_id does not match image/mask.")
if not source_check["image_exists"].all():
    display(source_check.loc[~source_check["image_exists"]].head(20))
    raise FileNotFoundError("Has image source not found.")
if not source_check["mask_exists"].all():
    display(source_check.loc[~source_check["mask_exists"]].head(20))
    raise FileNotFoundError("Has mask source not found.")

for left, right in (("train", "valid"), ("train", "test"), ("valid", "test")):
    overlap = all_ids[left] & all_ids[right]
    if overlap:
        raise ValueError(f"Leakage {left} ↔ {right}: {sorted(overlap)[:10]}")

source_validation = source_check.groupby("split").agg(
    rows=("image_id", "size"),
    mapping_ok=("mapping_ok", "all"),
    images_exist=("image_exists", "all"),
    masks_exist=("mask_exists", "all"),
)
display(source_validation)
print(
    "Success: Not found duplicated image_id across splits, and all source images/masks exist."
)

,rows,mapping_ok,images_exist,masks_exist
split,,,,
test,200,True,True,True
train,700,True,True,True
valid,100,True,True,True


Success: Not found duplicated image_id across splits, and all source images/masks exist.


# Distribution of label


In [7]:
split_counts = pd.DataFrame(
    {
        "images": {name: len(df) for name, df in splits.items()},
    }
)
split_counts["ratio_%_of_1000"] = (
    split_counts["images"].div(split_counts["images"].sum()).mul(100)
)
display(Markdown("### Image by split"))
display(split_counts.style.format({"ratio_%_of_1000": "{:.2f}%"}))

stratum_counts = (
    pd.concat(
        {name: df["stratum"].value_counts() for name, df in splits.items()},
        axis=1,
    )
    .fillna(0)
    .astype(int)
)
stratum_counts.columns.name = "split"
stratum_ratio = stratum_counts.div(stratum_counts.sum(axis=0), axis=1).mul(100)

display(Markdown("### Image-level label distribution - count"))
display(stratum_counts)
display(Markdown("### Image-level label distribution - ratio (%)"))
display(stratum_ratio.style.format("{:.2f}%"))

class_presence = pd.DataFrame(
    [
        {
            "split": name,
            "images": len(df),
            "images_with_non_neoplastic": int(df["has_non_neoplastic"].sum()),
            "non_neoplastic_presence_%": 100 * df["has_non_neoplastic"].mean(),
            "images_with_neoplastic": int(df["has_neoplastic"].sum()),
            "neoplastic_presence_%": 100 * df["has_neoplastic"].mean(),
            "multi_class_images": int((df["stratum"] == "both").sum()),
            "multi_class_%": 100 * (df["stratum"] == "both").mean(),
        }
        for name, df in splits.items()
    ]
).set_index("split")
display(Markdown("### Class presence and multi-class images"))
display(
    class_presence.style.format(
        {
            "non_neoplastic_presence_%": "{:.2f}%",
            "neoplastic_presence_%": "{:.2f}%",
            "multi_class_%": "{:.2f}%",
        }
    )
)

### Image by split

,images,ratio_%_of_1000
train,700,70.00%
valid,100,10.00%
test,200,20.00%


### Image-level label distribution - count

split,train,valid,test
stratum,,,
neo_only,486,69,139
non_neo_only,180,26,51
both,34,5,10


### Image-level label distribution - ratio (%)

split,train,valid,test
stratum,,,
neo_only,69.43%,69.00%,69.50%
non_neo_only,25.71%,26.00%,25.50%
both,4.86%,5.00%,5.00%


### Class presence and multi-class images

,images,images_with_non_neoplastic,non_neoplastic_presence_%,images_with_neoplastic,neoplastic_presence_%,multi_class_images,multi_class_%
split,,,,,,,
train,700,214,30.57%,520,74.29%,34,4.86%
valid,100,31,31.00%,74,74.00%,5,5.00%
test,200,61,30.50%,149,74.50%,10,5.00%


# Labeled subset in train


In [8]:
train_df = splits["train"]
label_rows = []

for column, expected in EXPECTED_LABELED.items():
    if not train_df[column].isin([0, 1]).all():
        raise ValueError(f"{column} must only contain 0/1.")

    labeled = train_df[train_df[column].eq(1)]
    unlabeled = train_df[train_df[column].eq(0)]
    if len(labeled) != expected:
        raise ValueError(
            f"{column}: expected {expected} labeled, received {len(labeled)}."
        )

    for status, group in (("labeled", labeled), ("unlabeled", unlabeled)):
        counts = group["stratum"].value_counts()
        label_rows.append(
            {
                "setting": column,
                "status": status,
                "images": len(group),
                "neo_only": int(counts.get("neo_only", 0)),
                "non_neo_only": int(counts.get("non_neo_only", 0)),
                "both": int(counts.get("both", 0)),
            }
        )

ids_1_4 = set(train_df.loc[train_df["labeled_1_4"].eq(1), "image_id"])
ids_1_2 = set(train_df.loc[train_df["labeled_1_2"].eq(1), "image_id"])
if not ids_1_4.issubset(ids_1_2):
    raise ValueError("labeled_1_4 is not a subset of labeled_1_2.")

label_summary = pd.DataFrame(label_rows)
display(label_summary)
print("Nested labeled subset 1/4 ⊂ 1/2: True")

,setting,status,images,neo_only,non_neo_only,both
0,labeled_1_4,labeled,175,122,45,8
1,labeled_1_4,unlabeled,525,364,135,26
2,labeled_1_2,labeled,350,243,90,17
3,labeled_1_2,unlabeled,350,243,90,17


Nested labeled subset 1/4 ⊂ 1/2: True


# Mask and pixel-level class


In [9]:
mask_ratio_by_split = {}
pixel_rows = []

for name, df in splits.items():
    total_pixels_per_image = df["width"] * df["height"]
    class_sum = df["background_pixels"] + df["non_neo_pixels"] + df["neo_pixels"]
    if not class_sum.eq(total_pixels_per_image).all():
        raise ValueError(f"{name}: Total pixel class does not equal width * height.")

    foreground = df["non_neo_pixels"] + df["neo_pixels"]
    mask_ratio = foreground / total_pixels_per_image
    mask_ratio_by_split[name] = mask_ratio

    total_background = int(df["background_pixels"].sum())
    total_non_neo = int(df["non_neo_pixels"].sum())
    total_neo = int(df["neo_pixels"].sum())
    total_all = total_background + total_non_neo + total_neo
    total_foreground = total_non_neo + total_neo

    pixel_rows.append(
        {
            "split": name,
            "background_pixels": total_background,
            "non_neo_pixels": total_non_neo,
            "neo_pixels": total_neo,
            "foreground_pixels": total_foreground,
            "foreground_%_of_all_pixels": 100 * total_foreground / total_all,
            "non_neo_%_of_foreground": 100 * total_non_neo / total_foreground,
            "neo_%_of_foreground": 100 * total_neo / total_foreground,
        }
    )

pixel_summary = pd.DataFrame(pixel_rows).set_index("split")
display(Markdown("### Pixel-level class distribution"))
display(
    pixel_summary.style.format(
        {
            "background_pixels": "{:,.0f}",
            "non_neo_pixels": "{:,.0f}",
            "neo_pixels": "{:,.0f}",
            "foreground_pixels": "{:,.0f}",
            "foreground_%_of_all_pixels": "{:.2f}%",
            "non_neo_%_of_foreground": "{:.2f}%",
            "neo_%_of_foreground": "{:.2f}%",
        }
    )
)

mask_ratio_summary = pd.DataFrame(
    {
        name: ratio.describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95])
        for name, ratio in mask_ratio_by_split.items()
    }
).T
display(Markdown("### Mask coverage ratio by image"))
display(mask_ratio_summary.style.format("{:.4f}"))

bins = [-np.inf, 0, 0.01, 0.05, 0.10, 0.25, 0.50, 1, np.inf]
labels = [
    "≤0",
    "(0,1%]",
    "(1,5%]",
    "(5,10%]",
    "(10,25%]",
    "(25,50%]",
    "(50,100%]",
    ">100%",
]
mask_ratio_bins = (
    pd.DataFrame(
        {
            name: pd.cut(ratio, bins=bins, labels=labels).value_counts(sort=False)
            for name, ratio in mask_ratio_by_split.items()
        }
    )
    .fillna(0)
    .astype(int)
)
display(Markdown("### Images by mask coverage interval"))
display(mask_ratio_bins)

### Pixel-level class distribution

,background_pixels,non_neo_pixels,neo_pixels,foreground_pixels,foreground_%_of_all_pixels,non_neo_%_of_foreground,neo_%_of_foreground
split,,,,,,,
train,"835,129,817","5,655,991","40,030,832","45,686,823",5.19%,12.38%,87.62%
valid,"119,740,948","1,269,472","4,744,460","6,013,932",4.78%,21.11%,78.89%
test,"237,505,016","1,395,803","12,690,861","14,086,664",5.60%,9.91%,90.09%


### Mask coverage ratio by image

,count,mean,std,min,5%,25%,50%,75%,95%,max
train,700.0000,0.0519,0.0536,0.0014,0.0065,0.0180,0.0356,0.0652,0.1705,0.4069
valid,100.0000,0.0479,0.0620,0.0043,0.0088,0.0186,0.0301,0.0521,0.1484,0.4411
test,200.0000,0.0561,0.0684,0.0022,0.0063,0.0155,0.0347,0.0698,0.1729,0.4581


### Images by mask coverage interval

,train,valid,test
≤0,0,0,0
"(0,1%]",70,8,28
"(1,5%]",392,65,106
"(5,10%]",148,18,34
"(10,25%]",82,7,27
"(25,50%]",8,2,5
"(50,100%]",0,0,0
>100%,0,0,0


# Struture main data folder


## Make `data/completed`


In [10]:
COMPLETED_DIR.mkdir(parents=True, exist_ok=True)
COMPLETED_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
COMPLETED_GT_DIR.mkdir(parents=True, exist_ok=True)
for directory in COMPLETED_IMAGE_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)


def copy_safely(source: Path, destination: Path) -> str:
    if not source.is_file():
        raise FileNotFoundError(source)

    if destination.exists():
        if filecmp.cmp(source, destination, shallow=False):
            return "already_identical"
        if not ALLOW_REPLACE_DIFFERENT_FILES:
            raise FileExistsError(
                f"File to be overwritten exists with different content: {destination}. "
                "Set ALLOW_REPLACE_DIFFERENT_FILES=True if you want to overwrite."
            )

    shutil.copy2(source, destination)
    return "copied"


copy_log = []
output_splits = {}

for split_name, df in splits.items():
    output_df = df.copy()

    for row_index, row in output_df.iterrows():
        image_id = str(row["image_id"])
        image_source = DATA_DIR / Path(str(row["image_path"]))
        mask_source = DATA_DIR / Path(str(row["mask_path"]))
        image_destination = COMPLETED_IMAGE_DIRS[split_name] / image_id
        mask_destination = COMPLETED_GT_DIR / image_id

        image_status = copy_safely(image_source, image_destination)
        mask_status = copy_safely(mask_source, mask_destination)

        output_df.at[row_index, "image_path"] = f"{split_name}/{image_id}"
        output_df.at[row_index, "mask_path"] = f"gt/{image_id}"

        copy_log.append(
            {
                "split": split_name,
                "image_id": image_id,
                "image_status": image_status,
                "mask_status": mask_status,
            }
        )

    output_splits[split_name] = output_df

copy_log_df = pd.DataFrame(copy_log)
display(
    copy_log_df.groupby("split").agg(
        images=("image_id", "size"),
        images_copied=("image_status", lambda s: int(s.eq("copied").sum())),
        images_already_identical=(
            "image_status",
            lambda s: int(s.eq("already_identical").sum()),
        ),
        masks_copied=("mask_status", lambda s: int(s.eq("copied").sum())),
        masks_already_identical=(
            "mask_status",
            lambda s: int(s.eq("already_identical").sum()),
        ),
    )
)

,images,images_copied,images_already_identical,masks_copied,masks_already_identical
split,,,,,
test,200,0,200,0,200
train,700,0,700,0,700
valid,100,0,100,0,100


## Overwrite 3 csv file


In [11]:
def write_csv_safely(df: pd.DataFrame, destination: Path) -> str:
    content = df.to_csv(index=False, lineterminator="\n")

    if destination.exists():
        current = destination.read_text(encoding="utf-8")
        if current == content:
            return "already_identical"
        if not ALLOW_REPLACE_DIFFERENT_FILES:
            raise FileExistsError(
                f"CSV to be overwritten exists with different content: {destination}. "
                "Set ALLOW_REPLACE_DIFFERENT_FILES=True if you want to overwrite."
            )

    destination.write_text(content, encoding="utf-8")
    return "successfully_written"


csv_status = {}
for split_name, output_df in output_splits.items():
    csv_destination = COMPLETED_SAMPLE_DIR / f"{split_name}.csv"
    csv_status[split_name] = write_csv_safely(output_df, csv_destination)

display(pd.DataFrame.from_dict(csv_status, orient="index", columns=["status"]))

,status
train,already_identical
valid,already_identical
test,already_identical


## Check


In [12]:
completed_splits = {
    name: pd.read_csv(COMPLETED_SAMPLE_DIR / f"{name}.csv") for name in SPLIT_NAMES
}

validation_rows = []
for split_name, df in completed_splits.items():
    expected_ids = set(df["image_id"].astype(str))
    actual_image_ids = {
        path.name
        for path in COMPLETED_IMAGE_DIRS[split_name].iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    }

    image_mapping_ok = df.apply(
        lambda row: (
            Path(str(row["image_path"])).parts[0] == split_name
            and Path(str(row["image_path"])).name == str(row["image_id"])
            and (COMPLETED_DIR / Path(str(row["image_path"]))).is_file()
        ),
        axis=1,
    ).all()
    mask_mapping_ok = df.apply(
        lambda row: (
            Path(str(row["mask_path"])).parts[0] == "gt"
            and Path(str(row["mask_path"])).name == str(row["image_id"])
            and (COMPLETED_DIR / Path(str(row["mask_path"]))).is_file()
        ),
        axis=1,
    ).all()

    validation_rows.append(
        {
            "split": split_name,
            "csv_rows": len(df),
            "expected_rows": EXPECTED_ROWS[split_name],
            "folder_images": len(actual_image_ids),
            "folder_matches_csv": actual_image_ids == expected_ids,
            "image_mapping_ok": bool(image_mapping_ok),
            "mask_mapping_ok": bool(mask_mapping_ok),
        }
    )

expected_all_ids = set().union(
    *(set(df["image_id"].astype(str)) for df in completed_splits.values())
)
actual_gt_ids = {
    path.name
    for path in COMPLETED_GT_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
}

validation_summary = pd.DataFrame(validation_rows).set_index("split")
display(validation_summary)

assert validation_summary["csv_rows"].eq(validation_summary["expected_rows"]).all()
assert validation_summary["folder_matches_csv"].all()
assert validation_summary["image_mapping_ok"].all()
assert validation_summary["mask_mapping_ok"].all()
assert actual_gt_ids == expected_all_ids
assert len(actual_gt_ids) == 1000
assert set(path.name for path in COMPLETED_SAMPLE_DIR.glob("*.csv")) == {
    "train.csv",
    "valid.csv",
    "test.csv",
}

final_summary = pd.DataFrame(
    {
        "item": [
            "train images",
            "valid images",
            "test images",
            "GT masks",
            "CSV files",
        ],
        "count": [
            len(completed_splits["train"]),
            len(completed_splits["valid"]),
            len(completed_splits["test"]),
            len(actual_gt_ids),
            len(list(COMPLETED_SAMPLE_DIR.glob("*.csv"))),
        ],
    }
)
display(final_summary)
display(
    Markdown(
        "## data/completed` has been successfully created with the following structure:"
    )
)

,csv_rows,expected_rows,folder_images,folder_matches_csv,image_mapping_ok,mask_mapping_ok
split,,,,,,
train,700,700,700,True,True,True
valid,100,100,100,True,True,True
test,200,200,200,True,True,True


,item,count
0,train images,700
1,valid images,100
2,test images,200
3,GT masks,1000
4,CSV files,3


## data/completed` has been successfully created with the following structure: